In [ ]:
import os
import gc
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, balanced_accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

In [ ]:
DATA_DIR = "../datasets/processed"


def load_all_datasets():
    paths = {
        "imdb": os.path.join(DATA_DIR, "imdb.csv"),
        "rotten": os.path.join(DATA_DIR, "rotten.csv"),
        "amazon": os.path.join(DATA_DIR, "amazon.csv"),
        "yelp": os.path.join(DATA_DIR, "yelp.csv"),
    }

    datasets = {}

    for name, path in paths.items():
        if not os.path.exists(path):
            raise FileNotFoundError(f"Nie znaleziono pliku: {path}")

        df = pd.read_csv(path)
        df = df[["text", "label"]].dropna()
        df["label"] = df["label"].astype(int)
        datasets[name] = df

    return datasets


datasets = load_all_datasets()

for domain, df in datasets.items():
    print(f"{domain}: {len(df)} samples")

In [ ]:
def compute_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }

In [ ]:
def run_ood_experiment_svm(
        datasets,
        n_splits=5,
        max_features=50_000,
        ngram_range=(1, 2)
):
    results = []
    training_logs = []

    domains = list(datasets.keys())

    for train_domain in domains:
        print("\n" + "="*80)
        print(f"TRAIN DOMAIN: {train_domain}")
        print("="*80)

        df = datasets[train_domain].reset_index(drop=True)
        X_texts = df["text"].values
        y = df["label"].values.astype(int)

        kf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

        for fold_id, (train_idx, val_idx) in enumerate(kf.split(X_texts, y), 1):
            print(f"\nFold {fold_id}/{n_splits}")

            X_train_texts = X_texts[train_idx]
            y_train = y[train_idx]

            X_val_texts = X_texts[val_idx]
            y_val = y[val_idx]

            vectorizer = TfidfVectorizer(
                max_features=max_features,
                ngram_range=ngram_range,
                sublinear_tf=True
            )
            X_train = vectorizer.fit_transform(X_train_texts)
            X_val = vectorizer.transform(X_val_texts)

            clf = LinearSVC(
                C=1.0,
                class_weight="balanced",
                max_iter=5000,
                random_state=42,
                dual="auto"
            )

            clf.fit(X_train, y_train)

            train_pred = clf.predict(X_train)
            val_pred = clf.predict(X_val)

            train_metrics = compute_metrics(y_train, train_pred)
            val_metrics = compute_metrics(y_val, val_pred)

            training_logs.append({
                "train_domain": train_domain,
                "fold": fold_id,
                "train_accuracy": train_metrics["accuracy"],
                "train_balanced_accuracy": train_metrics["balanced_accuracy"],
                "train_precision": train_metrics["precision"],
                "train_recall": train_metrics["recall"],
                "train_f1": train_metrics["f1"],
                "val_accuracy": val_metrics["accuracy"],
                "val_balanced_accuracy": val_metrics["balanced_accuracy"],
                "val_precision": val_metrics["precision"],
                "val_recall": val_metrics["recall"],
                "val_f1": val_metrics["f1"],
            })

            print(
                f"IND | F1: {val_metrics['f1']:.4f} | "
                f"Acc: {val_metrics['accuracy']:.4f} | "
                f"Prec: {val_metrics['precision']:.4f} | "
                f"Rec: {val_metrics['recall']:.4f}"
            )

            results.append({
                "train_domain": train_domain,
                "test_domain": train_domain,
                "fold": fold_id,
                "accuracy": val_metrics["accuracy"],
                "precision": val_metrics["precision"],
                "recall": val_metrics["recall"],
                "f1": val_metrics["f1"],
                "eval_type": "IND"
            })

            for test_domain, test_df in datasets.items():
                if test_domain == train_domain:
                    continue

                X_test_texts = test_df["text"].values
                y_test = test_df["label"].values.astype(int)

                X_test = vectorizer.transform(X_test_texts)
                test_pred = clf.predict(X_test)
                test_metrics = compute_metrics(y_test, test_pred)

                print(
                    f"OOD {train_domain} → {test_domain} | "
                    f"F1: {test_metrics['f1']:.4f} | "
                    f"Acc: {test_metrics['accuracy']:.4f} | "
                    f"Prec: {test_metrics['precision']:.4f} | "
                    f"Rec: {test_metrics['recall']:.4f}"
                )

                results.append({
                    "train_domain": train_domain,
                    "test_domain": test_domain,
                    "fold": fold_id,
                    "accuracy": test_metrics["accuracy"],
                    "precision": test_metrics["precision"],
                    "recall": test_metrics["recall"],
                    "f1": test_metrics["f1"],
                    "eval_type": "OOD"
                })

            del clf, vectorizer, X_train, X_val
            gc.collect()

    return pd.DataFrame(training_logs), pd.DataFrame(results)

In [ ]:
training_logs_svm, ood_results_svm = run_ood_experiment_svm(
    datasets=datasets,
    n_splits=5,
    max_features=50_000,
    ngram_range=(1, 2)
)

In [ ]:
SAVE_DIR = "./results"
os.makedirs(SAVE_DIR, exist_ok=True)

training_logs_svm.to_csv(os.path.join(SAVE_DIR, "svm_training_logs.csv"), index=False)
ood_results_svm.to_csv(os.path.join(SAVE_DIR, "svm_ood_results.csv"), index=False)

print("Saved:")
print(" - svm_training_logs.csv")
print(" - svm_ood_results.csv")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

pivot = ood_results_svm.groupby(
    ["train_domain", "test_domain"]
)["f1"].mean().unstack()

plt.figure(figsize=(8, 6))
sns.heatmap(pivot, annot=True, cmap="viridis", vmin=0, vmax=1)
plt.title("SVM+TF-IDF OOD F1")
plt.tight_layout()
plt.show()

In [ ]:
train_domains = ood_results_svm["train_domain"].unique()

for train_domain in train_domains:
    subset = ood_results_svm[ood_results_svm["train_domain"] == train_domain]
    
    pivot = subset.pivot_table(
        index="test_domain",
        columns="eval_type",
        values="accuracy",
        aggfunc="mean"
    )
    
    pivot = pivot.sort_index()
    
    plt.figure(figsize=(10, 5))
    ax = pivot.plot(kind="bar", width=0.8)
    
    plt.title(f"SVM+TF-IDF Accuracy (IND vs OOD) - trained on {train_domain}")
    plt.xlabel("Test domain")
    plt.ylabel("Accuracy")
    plt.xticks(rotation=45)
    plt.ylim(0, 1)
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()

plt.show()